In [ ]:
import pandas as pd
import numpy as np

# loading PubChemfingerprint file of EZH2 data

In [ ]:
df = pd.read_csv('EZH2_Active_Inactive_data.csv')
df

In [ ]:
df['bioactivity_class'].value_counts()

In [ ]:
w = df.drop('bioactivity_class', axis=1)
y = df.bioactivity_class
x = w.drop('Name', axis=1)

In [ ]:
names = df['Name']  # Keeping the names here for later reference

In [ ]:
y.replace(('active', 'inactive',),(1,0), inplace=True)
y

# variance based feature selection

In [ ]:
from sklearn.feature_selection import VarianceThreshold

def remove_low_variance(input_data, threshold=0.1):
    selection = VarianceThreshold(threshold)
    selection.fit(input_data)
    return input_data[input_data.columns[selection.get_support(indices=True)]]

x = remove_low_variance(x, threshold=0.1)
x

In [ ]:
x.to_csv('pubchem_reduced_fingerprint.csv', index = False) 
# Load the CSV files into DataFrames
df1 = pd.read_csv('pubchem_reduced_fingerprint.csv')

# loading a screening library into a dataframe

In [ ]:
#Maybridge_screening_library
molecule_library_df = pd.read_csv('maybridge_pubchemfingerprints.csv') #instead of maybridge_pubchemfingerprints, one can call their preffered liberary for screen
molecule_library_df

In [ ]:
screen_df = molecule_library_df.drop('Name', axis=1)
screen_df.to_csv('maybridge_withoutname_pubfingerprint.csv', index = False)
df2 = pd.read_csv('maybridge_withoutname_pubfingerprint.csv')

In [ ]:
# Get the list of columns in df1
columns_csv1 = set(df1.columns)

In [ ]:
# Extraction of variance based features identical to our EZH2 curated dataset
matching_columns = [col for col in df2.columns if col in columns_csv1]

In [ ]:
# Create a new DataFrame with only the matching columns
df2_filtered = df2[matching_columns]
df2_filtered

# External dataset processing.

In [ ]:
ext_data = pd.read_csv('external_dataset_descriptor.csv') 
ext_data['bioactivity_class'].value_counts()

In [ ]:
w1 = ext_data.drop('bioactivity_class', axis=1)
y1 = ext_data.bioactivity_class
x1 = w1.drop('Name', axis=1)

In [ ]:
y1.replace(('active', 'inactive',),(1,0), inplace=True)
y1

In [ ]:
ext_data.to_csv('externaldata_withoutname_fingerprints.csv', index = False)
# Load the CSV files into DataFrames
df1 = pd.read_csv('pubchem_reduced_fingerprint.csv')
df3 = pd.read_csv('externaldata_withoutname_fingerprints.csv')
# Get the list of columns in df1
columns_csv1 = set(df1.columns)
# Extract columns from CSV_2 that also exist in df3
matching_column = [col for col in df3.columns if col in columns_csv1]
# Create a new DataFrame with only the matching columns
df3_filtered = df3[matching_column]
df3_filtered

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y,test_size=.3,random_state =42)

In [ ]:
x_train.shape

In [ ]:
x_test.shape

In [ ]:
# Normalize features
scaler = MinMaxScaler()
x_train_normalized = scaler.fit_transform(x_train)
x_test_normalized = scaler.transform(x_test)

# saving a CSV file of training and test set

In [ ]:
# Create a results table for your test set
train_names = names.loc[x_train.index] # This gets names for only the test rows
results = pd.DataFrame({
    'Molecule_Name': train_names,
    'Actual': y_train,
    #'Predicted': svm_tuned_pred
})

print(results.head())

In [ ]:
train_names.to_csv('train_set_names.csv', index = False)
train_names

In [ ]:
# Create a results table for your test set
test_names = names.loc[x_test.index] # This gets names for only the test rows
results = pd.DataFrame({
    'Molecule_Name': test_names,
    'Actual': y_test,
    #'Predicted': svm_tuned_pred
})

print(results.head())

In [ ]:
test_names.to_csv('test_set_names.csv', index = False)
test_names

In [ ]:
# Convert to same type and verify
df_all = pd.read_csv('EZH2_smiles_data.csv')
print((df_all['name'].astype(str).values == df['Name'].astype(str).values).all())

In [ ]:
# Create mapping: df gapped index -> df_all clean index
index_map = dict(zip(df.index, df_all.index))

# Map x_train and x_test indices
train_mapped = [index_map[i] for i in x_train.index]
test_mapped = [index_map[i] for i in x_test.index]

ezh2_train_set = df_all.loc[train_mapped]
ezh2_test_set = df_all.loc[test_mapped]

print(f"x_train: {len(x_train)} | Train CSV: {len(ezh2_train_set)}")
print(f"x_test: {len(x_test)} | Test CSV: {len(ezh2_test_set)}")

# Save
ezh2_train_set.to_csv('EZH2_train_set.csv', index=False)
ezh2_test_set.to_csv('EZH2_test_set.csv', index=False)

In [ ]:
ezh2_train_set['bioactivity_class'].value_counts()

In [ ]:
ezh2_test_set['bioactivity_class'].value_counts()

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import ExtraTreesClassifier

In [ ]:
# Initialize classifiers
svm = SVC()
rf = RandomForestClassifier()
xgb = XGBClassifier()
knn = KNeighborsClassifier()
etc = ExtraTreesClassifier()

# Hyperparameter tuning for SVM

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
# Define the parameter grid to search
param_grid = {
    'kernel': ['linear', 'rbf', 'poly'],  # Types of kernels to try
    'C': [100, 1000],  # Values of C to try
    'gamma': ['scale', 'auto', 0.0001, 0.001, 0.01],  # Values of gamma to try (for 'rbf' and 'poly' kernels)
    #'class_weight': [None, 'balanced', {0: 1, 1: 2}, {0: 1, 1: 5}]  # Class weights to try
}

In [ ]:
# Create GridSearchCV object
grid_search = GridSearchCV(estimator=svm, param_grid=param_grid, cv=5, scoring='accuracy', verbose=2)
#(estimator=svm_classifier is already been called above)

In [ ]:
#Perform hyperparameter tuning
grid_search.fit(x_train_normalized, y_train)

In [ ]:
# Print best parameters found
print("Best Parameters:", grid_search.best_params_)

In [ ]:
# Get the best model
best_svm = grid_search.best_estimator_

In [ ]:
best_svm = SVC(kernel = 'poly', C = 100, gamma = 0.01) #hypertuned parameter based on grid search

In [ ]:
best_svm.fit(x_train, y_train)

In [ ]:
# Evaluate the best model
svm_tuned_pred = best_svm.predict(x_test_normalized)
svm_tuned_accuracy = accuracy_score(y_test, svm_tuned_pred)
from sklearn.metrics import classification_report
svm_report = classification_report(y_test, svm_tuned_pred)

In [ ]:
# Print evaluation results
print("Evaluation Results:")
print(f"Accuracy: {svm_tuned_accuracy}")
print("Classification Report:")
print(svm_report)

In [ ]:
svm_tuned_cm = confusion_matrix(y_test,svm_tuned_pred)
print(svm_tuned_cm)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

In [ ]:
fpr, tpr, _ = roc_curve(y_test, svm_tuned_pred)
# Calculate AUC
roc_auc = auc(fpr, tpr)
# Plot confusion matrix
plt.figure(figsize=(5, 5))
sns.heatmap(svm_tuned_cm, annot=True, fmt='d', cmap='Blues', cbar=False, annot_kws={"size": 16})
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('SVM_test_set_Confusion_Matrix')
#plt.savefig("SVM_test_set_confusion_matrix.jpg", format="jpg")
plt.show()

In [ ]:
# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label='ROC curve (AUC = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([-0.1, 1.0])
plt.ylim([-0.1, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
#plt.savefig("SVM_roc_curve_comparison_test_set.jpg", format="jpg")
plt.show()

# External data validation for svm

In [ ]:
# Evaluate the best model
svm_extern_pred = best_svm.predict(df3_filtered)
svm_extern_accuracy = accuracy_score(y1, svm_extern_pred)
from sklearn.metrics import classification_report
svm_extern_report = classification_report(y1, svm_extern_pred)

In [ ]:
# Print evaluation results
print("Evaluation Results:")
print(f"Accuracy: {svm_extern_accuracy}")
print("Classification Report:")
print(svm_extern_report)

In [ ]:
#confusion matrix of external data set
svm_extern_cm = confusion_matrix(y1,svm_extern_pred)
print(svm_extern_cm)
fpr, tpr, _ = roc_curve(y1, svm_extern_pred)
# Calculate AUC
roc_auc = auc(fpr, tpr)

In [ ]:
# Plot confusion matrix
plt.figure(figsize=(5, 5))
sns.heatmap(svm_extern_cm, annot=True, fmt='d', cmap='Blues', cbar=False, annot_kws={"size": 16})
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('SVM_validation_ConfusionMatrix')
#plt.savefig("svm_external_set_confusion_matrix.jpg", format="jpg")
plt.show()

In [ ]:
# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label='ROC curve (AUC = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([-0.1, 1.0])
plt.ylim([-0.1, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
#plt.savefig("SVM_roc_curve_external_set.jpg", format="jpg")
plt.show()

# Hyperparameter tuning of Random Forest

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
# Define parameter grid for hyperparameter tuning
param_grid = {
    'n_estimators': [int(x) for x in np.linspace(start = 200, stop = 2000, num = 10)],
    'max_features': ['auto', 'sqrt'],
    'max_depth': [int(x) for x in np.linspace(10, 110, num = 11)] + [None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

In [ ]:
# Perform randomized search with 5-fold cross-validation
random_search = RandomizedSearchCV(estimator=rf, param_distributions=param_grid, n_iter=100, cv=5, scoring='accuracy', verbose=2, random_state=42, n_jobs=-1)
random_search.fit(x_train_normalized, y_train)

In [ ]:
# Get best hyperparameters
rf_tuned = random_search.best_params_
print("Best hyperparameters:", rf_tuned)

In [ ]:
rf_tuned = RandomForestClassifier(n_estimators = 1000, min_samples_split = 2, min_samples_leaf = 1, max_features = 'sqrt', max_depth = 20,
                                 bootstrap = True) #hypertuned parameters based on random_search 

In [ ]:
rf_tuned.fit(x_train_normalized, y_train)

In [ ]:
# Evaluate best model on test set
rf_pred = rf_tuned.predict(x_test_normalized)
rf_tuned_accuracy = accuracy_score(y_test, rf_pred)
rf_report = classification_report(y_test, rf_pred)
print("Test accuracy:", rf_tuned_accuracy)

In [ ]:
rf_tuned_cm = confusion_matrix(y_test,rf_pred)
print(rf_tuned_cm)

In [ ]:
fpr, tpr, _ = roc_curve(y_test, rf_pred)
# Calculate AUC
roc_auc = auc(fpr, tpr)

In [ ]:
# Plot confusion matrix
plt.figure(figsize=(5, 5))
sns.heatmap(rf_tuned_cm, annot=True, fmt='d', cmap='Blues', cbar=False, annot_kws={"size": 16})
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('RF_test_set_ConfusionMatrix')
#plt.savefig("rf_test_set_confusion_matrix.jpg", format="jpg")
plt.show()

In [ ]:
# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label='ROC curve (AUC = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([-0.1, 1.0])
plt.ylim([-0.1, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
#plt.savefig("RF_roc_curve_test_set.jpg", format="jpg")
plt.show()

# External data validation of RF

In [ ]:
# Evaluate the best model
rf_extern_pred = rf_tuned.predict(df3_filtered)
rf_extern_accuracy = accuracy_score(y1, rf_extern_pred)
from sklearn.metrics import classification_report
rf_extern_report = classification_report(y1, rf_extern_pred)

In [ ]:
# Print evaluation results
print("Evaluation Results:")
print("Accuracy:", rf_extern_accuracy)
print("Classification Report:")
print(rf_extern_report)

In [ ]:
#confusion matrix of external data set
rf_extern_cm = confusion_matrix(y1,rf_extern_pred)
print(rf_extern_cm)

In [ ]:
# Plot confusion matrix
plt.figure(figsize=(5, 5))
sns.heatmap(rf_extern_cm, annot=True, fmt='d', cmap='Blues', cbar=False, annot_kws={"size": 16})
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('RF_external_set_ConfusionMatrix')
#plt.savefig("rf_external_set_confusion_matrix.jpg", format="jpg")
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y1, rf_extern_pred)
# Calculate AUC
roc_auc = auc(fpr, tpr)
# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label='ROC curve (AUC = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([-0.1, 1.0])
plt.ylim([-0.1, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
#plt.savefig("RF_roc_curve_EXTERNAL_set.jpg", format="jpg")
plt.show()

# Hyperparameter tuning of XGBoost

In [ ]:
# Define parameter grid for hyperparameter tuning
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.1, 0.01],
    'gamma': [0, 0.1, 0.2],
}

In [ ]:
# Perform grid search with 5-fold cross-validation
grid_search = GridSearchCV(estimator=xgb, param_grid=param_grid, cv=5, scoring='accuracy', verbose=2, n_jobs=-1)
grid_search.fit(x_train_normalized, y_train)

In [ ]:
# Get best hyperparameters
best_xgb = grid_search.best_params_
print("Best hyperparameters:", best_xgb)

In [ ]:
# Get best model
best_xgb = grid_search.best_estimator_
best_xgb = XGBClassifier(gamma= 0.1, learning_rate= 0.1, max_depth= 7) #hypertuned parameter based on grid search
best_xgb.fit(x_train_normalized, y_train)

In [ ]:
# Evaluate best model on test set
xgb_pred = best_xgb.predict(x_test_normalized)
xgb_accuracy = accuracy_score(y_test, xgb_pred)
xgb_report = classification_report(y_test, xgb_pred)

In [ ]:
# Print evaluation results
print("Evaluation Results:")
print("Accuracy:", xgb_accuracy)
print("Classification Report:")
print(xgb_report)

In [ ]:
xg_tuned_cm = confusion_matrix(y_test,xgb_pred)
print(xg_tuned_cm)

In [ ]:
# Plot confusion matrix
plt.figure(figsize=(5, 5))
sns.heatmap(xg_tuned_cm, annot=True, fmt='d', cmap='Blues', cbar=False, annot_kws={"size": 16})
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('XGB_TEST_set_ConfusionMatrix')
#plt.savefig("XGB_TEST_set_confusion_matrix.jpg", format="jpg")
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, xgb_pred)
# Calculate AUC
roc_auc = auc(fpr, tpr)
# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label='ROC curve (AUC = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([-0.1, 1.0])
plt.ylim([-0.1, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
#plt.savefig("XGB_roc_curve_TEST_set.jpg", format="jpg")
plt.show()

# External Data Validation of XGboost

In [ ]:
# Evaluate the best model
xgb_extern_pred = best_xgb.predict(df3_filtered)
xgb_extern_accuracy = accuracy_score(y1, xgb_extern_pred)
from sklearn.metrics import classification_report
xgb_extern_report = classification_report(y1, xgb_extern_pred)

In [ ]:
# Print evaluation results
print("Evaluation Results:")
print("Accuracy:", xgb_extern_accuracy)
print("Classification Report:")
print(xgb_extern_report)


In [ ]:
#confusion matrix of external data set
xgb_extern_cm = confusion_matrix(y1,xgb_extern_pred)
print(xgb_extern_cm)

In [ ]:
# Plot confusion matrix
plt.figure(figsize=(5, 5))
sns.heatmap(rf_extern_cm, annot=True, fmt='d', cmap='Blues', cbar=False, annot_kws={"size": 16})
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('XGB_EXTERNAL_set_ConfusionMatrix')
#plt.savefig("XGB_EXTERN_set_confusion_matrix.jpg", format="jpg")
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y1, xgb_extern_pred)
# Calculate AUC
roc_auc = auc(fpr, tpr)
# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label='ROC curve (AUC = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([-0.1, 1.0])
plt.ylim([-0.1, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
#plt.savefig("XGB_roc_curve_EXTERN_set.jpg", format="jpg")
plt.show()

# Hyperparameter tuning of KNN

In [ ]:
# Define hyperparameters grid for tuning
param_grid = {
    'n_neighbors': [3, 5, 7, 9],  # Number of neighbors
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],  # Algorithm used to compute nearest neighbors
    'leaf_size': [10, 20, 30, 40],  # Leaf size passed to BallTree or KDTree
}

In [ ]:
# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=knn, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)

In [ ]:
# Perform grid search
grid_search.fit(x_train_normalized, y_train)

In [ ]:
# Print best parameters found
print("Best Parameters:", grid_search.best_params_)

In [ ]:
# Get the best model
#best_knn = grid_search.best_estimator_
knn = KNeighborsClassifier(algorithm = 'auto', leaf_size = 10, n_neighbors = 3)

In [ ]:
knn.fit(x_train_normalized, y_train)

In [ ]:
# Evaluate the best model
knn_tuned_pred = knn.predict(x_test_normalized)
knn_tuned_accuracy = accuracy_score(y_test, knn_tuned_pred)
report = classification_report(y_test, knn_tuned_pred)

In [ ]:
# Print evaluation results
print("Evaluation Results:")
print("Accuracy:", knn_tuned_accuracy)
print("Classification Report:")
print(report)

In [ ]:
# Assuming y_test is the true labels for your test dataset
knn_tuned_cm = confusion_matrix(y_test, knn_tuned_pred)
print(knn_tuned_cm)

In [ ]:
# Plot confusion matrix
plt.figure(figsize=(5, 5))
sns.heatmap(knn_tuned_cm, annot=True, fmt='d', cmap='Blues', cbar=False, annot_kws={"size": 16})
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('KNN_TEST_set_ConfusionMatrix')
#plt.savefig("KNN_TEST_set_confusion_matrix.jpg", format="jpg")
plt.show()

In [ ]:
# Calculate ROC curve
fpr, tpr, _ = roc_curve(y_test, knn_tuned_pred)
# Calculate AUC
roc_auc = auc(fpr, tpr)
# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label='ROC curve (AUC = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([-0.1, 1.0])
plt.ylim([-0.1, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
#plt.savefig("knn_roc_curve_test_set.jpg", format="jpg")
plt.show()

In [ ]:
# Evaluate the best model
knn_extern_pred = knn.predict(df3_filtered)
knn_extern_accuracy = accuracy_score(y1, knn_extern_pred)
from sklearn.metrics import classification_report
knn_extern_report = classification_report(y1, knn_extern_pred)# Evaluate the best model

In [ ]:
# Print evaluation results
print("Evaluation Results:")
print("Accuracy:", knn_extern_accuracy)
print("Classification Report:")
print(knn_extern_report)

In [ ]:
#confusion matrix of external data set
knn_extern_cm = confusion_matrix(y1,knn_extern_pred)
print(knn_extern_cm)
# Plot confusion matrix
plt.figure(figsize=(5, 5))
sns.heatmap(knn_extern_cm, annot=True, fmt='d', cmap='Blues', cbar=False, annot_kws={"size": 16})
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('KNN_external_set_ConfusionMatrix')
#plt.savefig("KNN_external_set_confusion_matrix.jpg", format="jpg")
plt.show()

In [ ]:
# Calculate ROC curve
fpr, tpr, _ = roc_curve(y1,knn_extern_pred)
# Calculate AUC
roc_auc = auc(fpr, tpr)
# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label='ROC curve (AUC = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([-0.1, 1.0])
plt.ylim([-0.1, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
#plt.savefig("knn_roc_curve_external_set.jpg", format="jpg")
plt.show()

# KNN Screening of Maybridge Library

In [ ]:
#calling screening library in SMILE-csv file
maybridge_liberary = pd.read_csv('MAYBRIDGE_LIBERARY.csv')

In [ ]:
knn_screen_predictions = knn.predict(df2_filtered)
maybridge_liberary['knn_pubchem_pred'] = knn_screen_predictions
maybridge_liberary['knn_pubchem_pred'].value_counts()

In [ ]:
maybridge_liberary.to_csv('Pubchem_MODEL_prediction.csv', index=False)
data = pd.read_csv('Pubchem_MODEL_prediction.csv')
#extracting the rf actives from the smiles file of screened liberary
active_rows = data[data['knn_pubchem_pred'] == 1]
#creating the separate csv file of actives
active_rows.to_csv('KNN_PUBCHEM_actives.csv', index=False)